# Colab2: T4 GPU Training and Ensemble Prediction

This notebook is for Google Colab with a `T4 GPU`.

It will:
1. Verify GPU runtime
2. Clone the project
3. Install dependencies
4. Upload the new dataset
5. Optionally upload the ISOT-trained checkpoint zip
6. Train a new model on `Fake or Real News Dataset`
7. Evaluate it
8. Run ensemble prediction with two checkpoints


## Step 0: Select T4 GPU

In Colab:
- `Runtime` -> `Change runtime type`
- Hardware accelerator -> `T4 GPU`
- Save


In [1]:
import os
import sys
import torch

print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled. Switch Colab runtime to T4 GPU first.')

print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

PROJECT_DIR = '/content/FND_2027'
print('Project dir:', PROJECT_DIR)


Torch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory (GB): 15.64
Project dir: /content/FND_2027


In [2]:
!git clone https://github.com/au510621104021/FND_2027.git /content/FND_2027
%cd /content/FND_2027
!git branch --show-current
!git log -1 --oneline


Cloning into '/content/FND_2027'...
remote: Enumerating objects: 189, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 189 (delta 30), reused 35 (delta 11), pack-reused 117 (from 1)
Receiving objects: 100% (189/189), 57.83 MiB | 23.69 MiB/s, done.
Resolving deltas: 100% (73/73), done.
/content/FND_2027
main
3ba5a32 (HEAD -> main, origin/main, origin/HEAD) Add Colab model export workflow


In [3]:
!pip install -q --upgrade pip
!pip install -q -r requirements.txt
print('Dependencies installed')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 31.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Dependencies installed


## Step 1: Prepare folders

This notebook trains the second model on `Fake or Real News Dataset`.

Expected files:
- `train.csv`
- `test.csv`

Optional ISOT model zip:
- `trained_model.zip`


In [4]:
import os

SECOND_DATASET_DIR = os.path.join(PROJECT_DIR, 'Fake or Real News Dataset')
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
SECOND_CHECKPOINT_DIR = os.path.join(CHECKPOINT_DIR, 'fake_or_real')

os.makedirs(SECOND_DATASET_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SECOND_CHECKPOINT_DIR, exist_ok=True)

print('Second dataset dir:', SECOND_DATASET_DIR)
print('Checkpoint dir:', CHECKPOINT_DIR)
print('Second checkpoint dir:', SECOND_CHECKPOINT_DIR)


Second dataset dir: /content/FND_2027/Fake or Real News Dataset
Checkpoint dir: /content/FND_2027/checkpoints
Second checkpoint dir: /content/FND_2027/checkpoints/fake_or_real


## Step 2: Upload `train.csv` and `test.csv`

Run the next cell, upload the files for `Fake or Real News Dataset`, and they will be copied into the correct project folder.


In [5]:
from google.colab import files
import shutil

uploaded = files.upload()

for name in uploaded.keys():
    dest = os.path.join(SECOND_DATASET_DIR, name)
    shutil.move(name, dest)
    print('Saved:', dest)

print('Final files:', sorted(os.listdir(SECOND_DATASET_DIR)))


Saving test.csv to test.csv
Saving train.csv to train.csv
Saved: /content/FND_2027/Fake or Real News Dataset/test.csv
Saved: /content/FND_2027/Fake or Real News Dataset/train.csv
Final files: ['test.csv', 'train.csv']


In [6]:
required = ['train.csv', 'test.csv']
missing = [name for name in required if not os.path.exists(os.path.join(SECOND_DATASET_DIR, name))]
if missing:
    raise FileNotFoundError(f'Missing dataset files: {missing}')
print('Dataset files are ready')


Dataset files are ready


## Step 3: Optional - Upload your ISOT-trained model zip

If you want to use ensemble prediction in Colab, upload `trained_model.zip` here.

If you skip this step, training of the new model still works.


In [7]:
UPLOAD_ISOT_ZIP = False
print('Set UPLOAD_ISOT_ZIP = True if you want to upload the old ISOT checkpoint zip now.')


Set UPLOAD_ISOT_ZIP = True if you want to upload the old ISOT checkpoint zip now.


In [8]:
import zipfile
import shutil

if UPLOAD_ISOT_ZIP:
    uploaded = files.upload()
    zip_names = [name for name in uploaded.keys() if name.lower().endswith('.zip')]
    if not zip_names:
        raise FileNotFoundError('No zip file uploaded.')

    isot_zip = zip_names[0]
    print('Uploaded zip:', isot_zip)

    with zipfile.ZipFile(isot_zip, 'r') as zf:
        zf.extractall(PROJECT_DIR)

    print('Extracted zip into project directory')
    print('Checkpoint files:', os.listdir(CHECKPOINT_DIR))
else:
    print('Skipping ISOT checkpoint upload')


Skipping ISOT checkpoint upload


## Step 4: Tune config for T4 GPU

This cell updates the second-dataset config for Colab.


In [9]:
import yaml

config_path = os.path.join(PROJECT_DIR, 'config', 'config_fake_or_real.yaml')
with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

config['data']['data_dir'] = SECOND_DATASET_DIR
config['data'].pop('data_dirs', None)
config['data']['num_workers'] = 2
config['data']['pin_memory'] = True

config['training']['batch_size'] = 4
config['training']['fp16'] = True
config['training']['num_epochs'] = 10

config['logging']['checkpoint_dir'] = './checkpoints/fake_or_real'
config['logging']['log_dir'] = './logs/fake_or_real'
config['inference']['model_checkpoint'] = './checkpoints/fake_or_real/best_model.pt'
config['publication']['results_dir'] = './results/fake_or_real'

with open(config_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(config, f, sort_keys=False)

print('Updated config at:', config_path)
print('Batch size:', config['training']['batch_size'])
print('Epochs:', config['training']['num_epochs'])
print('Data dir:', config['data']['data_dir'])


Updated config at: /content/FND_2027/config/config_fake_or_real.yaml
Batch size: 4
Epochs: 10
Data dir: /content/FND_2027/Fake or Real News Dataset


## Step 5: Train the new model on T4 GPU

This will create:
- `checkpoints/fake_or_real/best_model.pt`
- `checkpoints/fake_or_real/latest_model.pt`


In [10]:
%cd /content/FND_2027
!python scripts/train.py --config config/config_fake_or_real.yaml --device cuda


/content/FND_2027
2026-04-02 14:03:22.067692: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775138602.088048    2631 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775138602.094789    2631 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775138602.112698    2631 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775138602.112720    2631 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775138602.112724    2631 computation_placer.cc:177] comp

## Step 6: Evaluate the new model


In [11]:
%cd /content/FND_2027
!python scripts/evaluate.py --checkpoint checkpoints/fake_or_real/best_model.pt --config config/config_fake_or_real.yaml --device cuda --ablation --bootstrap --latex


/content/FND_2027
2026-04-02 14:34:27.949861: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775140467.988746   11110 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775140468.000317   11110 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775140468.028736   11110 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775140468.028774   11110 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775140468.028783   11110 computation_placer.cc:177] comp

## Step 7: Upload an image for prediction

You can test:
- the old ISOT model alone
- the new model alone
- both models together with ensemble averaging


In [12]:
uploaded = files.upload()
image_name = next(iter(uploaded.keys()))
IMAGE_PATH = os.path.join(PROJECT_DIR, image_name)
print('Image path:', IMAGE_PATH)


Saving Screenshot 2026-04-01 173136.png to Screenshot 2026-04-01 173136.png
Image path: /content/FND_2027/Screenshot 2026-04-01 173136.png


In [13]:
TEXT_INPUT = 'Enter the news text that belongs to the uploaded image here.'
print(TEXT_INPUT)


Enter the news text that belongs to the uploaded image here.


## Step 8A: Predict with ISOT model only

Run this only if `checkpoints/best_model.pt` exists.


In [14]:
!python scripts/predict.py --checkpoint checkpoints/best_model.pt --text "$TEXT_INPUT" --image "$IMAGE_PATH" --device cuda


Traceback (most recent call last):
  File "/content/FND_2027/scripts/predict.py", line 84, in <module>
    main()
  File "/content/FND_2027/scripts/predict.py", line 45, in main
    predictor = MultimodalPredictor.from_checkpoint(args.checkpoint, device=device)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/FND_2027/src/inference/predictor.py", line 85, in from_checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 1500, in load
    with _open_file_like(f, "rb") as opened_file:
         ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 768, in _open_file_like
    return _open_file(name_or_buffer, mode)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", 

## Step 8B: Predict with the new dataset model only


In [15]:
!python scripts/predict.py --checkpoint checkpoints/fake_or_real/best_model.pt --text "$TEXT_INPUT" --image "$IMAGE_PATH" --device cuda


Traceback (most recent call last):
  File "/content/FND_2027/scripts/predict.py", line 84, in <module>
    main()
  File "/content/FND_2027/scripts/predict.py", line 45, in main
    predictor = MultimodalPredictor.from_checkpoint(args.checkpoint, device=device)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/FND_2027/src/inference/predictor.py", line 85, in from_checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 1548, in load
    raise pickle.UnpicklingError(_get_wo_message(str(e))) from None
_pickle.UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from 

## Step 8C: Ensemble prediction with both models

Run this only if both checkpoints exist.


In [16]:
!python scripts/predict_ensemble.py --text "$TEXT_INPUT" --image "$IMAGE_PATH" --checkpoints checkpoints/best_model.pt checkpoints/fake_or_real/best_model.pt --device cuda


Traceback (most recent call last):
  File "/content/FND_2027/scripts/predict_ensemble.py", line 61, in <module>
    main()
  File "/content/FND_2027/scripts/predict_ensemble.py", line 35, in main
    ensemble = EnsemblePredictor.from_checkpoints(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/FND_2027/src/inference/ensemble.py", line 36, in from_checkpoints
    predictors = [MultimodalPredictor.from_checkpoint(path, device=device) for path in checkpoints]
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/FND_2027/src/inference/predictor.py", line 85, in from_checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 1500, in load
    with _open_file_like(f, "rb") as opened_file:
         ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch

## Step 9: Save trained second-model checkpoint

Download the new checkpoint after training.


In [17]:
from google.colab import files

files.download('/content/FND_2027/checkpoints/fake_or_real/best_model.pt')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 10: Export the trained second model as a zip

This creates a zip similar to your earlier trained-model archive.


In [18]:
!rm -rf /content/FND_2027/export_fake_or_real
!mkdir -p /content/FND_2027/export_fake_or_real/checkpoints
!cp /content/FND_2027/checkpoints/fake_or_real/best_model.pt /content/FND_2027/export_fake_or_real/checkpoints/
!cp /content/FND_2027/checkpoints/fake_or_real/latest_model.pt /content/FND_2027/export_fake_or_real/checkpoints/
!cp /content/FND_2027/checkpoints/fake_or_real/training_history.json /content/FND_2027/export_fake_or_real/checkpoints/
%cd /content/FND_2027
!rm -f fake_or_real_trained_model.zip
!zip -r fake_or_real_trained_model.zip export_fake_or_real


/content/FND_2027
  adding: export_fake_or_real/ (stored 0%)
  adding: export_fake_or_real/checkpoints/ (stored 0%)
  adding: export_fake_or_real/checkpoints/training_history.json (deflated 72%)
  adding: export_fake_or_real/checkpoints/latest_model.pt (deflated 8%)
  adding: export_fake_or_real/checkpoints/best_model.pt (deflated 8%)


In [19]:
from google.colab import files
files.download('/content/FND_2027/fake_or_real_trained_model.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>